# Kaggle Submission Notebook — Retrieval Engine Competition

**Group:** SeaFour  
**Pipeline:** Load data → Preprocess → Retrieve (BM25+ / TF-IDF / Embedding / Hybrid) → Evaluate on train → Generate submission CSV

This notebook implements four retrieval methods:
1. **TF-IDF** — sparse lexical baseline with bigrams and cosine similarity
2. **BM25+** — probabilistic lexical model with length normalisation
3. **Embedding Search** — dense semantic retrieval using Sentence-Transformers
4. **Hybrid (BM25+ → Embedding Re-ranking)** — BM25+ retrieves candidates, embedding model re-scores them, scores are fused

We evaluate every method on the training queries (MAP@100, Recall@100), then produce the final Kaggle submission CSV with the best-performing model.

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
from pathlib import Path
import csv, json, re, os, time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Data directory auto-detection (Kaggle → local fallback) ─────────────────
DATA_DIR = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'docs.json' in filenames:
        DATA_DIR = Path(dirname)
        break

if DATA_DIR is None:
    DATA_DIR = Path('../data')
    if not DATA_DIR.exists():
        raise FileNotFoundError(
            "Could not find data directory. "
            "On Kaggle, add the competition dataset; "
            "locally, place files in ../data/."
        )

print(f"Data directory : {DATA_DIR}")

# ── Tuneable parameters ─────────────────────────────────────────────────────
OUTPUT_PATH     = Path('solutions_SeaFour.csv')
TOP_K           = 100
FINAL_MODEL     = 'hybrid'         # 'bm25', 'tfidf', 'embedding', or 'hybrid'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'   # lightweight sentence-transformer
EMBEDDING_BATCH = 256

# ── Hybrid-specific parameters ──────────────────────────────────────────────
HYBRID_BM25_CANDIDATES = 500      # BM25+ retrieves this many candidates per query
HYBRID_ALPHA           = 0.35     # weight for BM25+ score in fusion (1-α for embedding)

Using data directory: ../data


In [ ]:
# ── 2. Shared Preprocessing ──────────────────────────────────────────────────

def value_to_text(value):
    """Normalise None / NaN / list / tuple → string."""
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)


def create_content_column(df, columns):
    """Build a single lowercase 'content' field from *columns*."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''
    merged = []
    for _, row in out[columns].iterrows():
        text = ' '.join(value_to_text(row[col]) for col in columns).strip().lower()
        merged.append(text)
    out['content'] = merged
    out['id'] = out['id'].astype(str)
    return out


_TOKEN_RE = re.compile(r'[a-z0-9]+')

def tokenize(text):
    """Lowercase tokeniser for BM25+ (normalises separators)."""
    txt = str(text or '').lower()
    txt = re.sub(r'[-_/]', ' ', txt)
    return _TOKEN_RE.findall(txt)

In [ ]:
# ── 3. Retrieval Methods ─────────────────────────────────────────────────────

# ── 3a. TF-IDF ──────────────────────────────────────────────────────────────
def run_tfidf_search(docs_df, queries_df, top_k=100):
    """TF-IDF with unigram+bigram features and cosine similarity."""
    top_k = min(top_k, len(docs_df))
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
    try:
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    except ValueError as err:
        if 'After pruning, no terms remain' not in str(err):
            raise
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
        doc_vectors = vectorizer.fit_transform(docs_df['content'])

    query_vectors = vectorizer.transform(queries_df['content'])
    scores = cosine_similarity(query_vectors, doc_vectors)
    doc_ids = docs_df['id'].to_numpy()

    results = []
    for i, row_scores in enumerate(scores):
        top_idx = np.argsort(row_scores)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


# ── 3b. BM25+ ───────────────────────────────────────────────────────────────
def run_bm25_search(docs_df, queries_df, top_k=100):
    """BM25+ over tokenised content."""
    from rank_bm25 import BM25Plus

    top_k = min(top_k, len(docs_df))
    tokenized_corpus = [tokenize(text) for text in docs_df['content']]
    bm25 = BM25Plus(tokenized_corpus)
    doc_ids = docs_df['id'].to_numpy()

    results = []
    for _, row in queries_df.iterrows():
        query_tokens = tokenize(row['content'])
        scores = bm25.get_scores(query_tokens)
        top_idx = np.argsort(scores)[-top_k:][::-1]
        results.append({
            'query_id': row['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


# ── 3c. Embedding Search ────────────────────────────────────────────────────
def run_embedding_search(docs_df, queries_df, top_k=100,
                         model_name=EMBEDDING_MODEL,
                         batch_size=EMBEDDING_BATCH):
    """Dense retrieval using a Sentence-Transformer model."""
    from sentence_transformers import SentenceTransformer

    top_k = min(top_k, len(docs_df))
    model = SentenceTransformer(model_name)

    print(f"  Encoding {len(docs_df)} documents …")
    doc_embeddings = model.encode(
        docs_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"  Encoding {len(queries_df)} queries …")
    query_embeddings = model.encode(
        queries_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    # cosine similarity (vectors are already L2-normalised → dot product)
    scores = query_embeddings @ doc_embeddings.T
    doc_ids = docs_df['id'].to_numpy()

    results = []
    for i, row_scores in enumerate(scores):
        top_idx = np.argsort(row_scores)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


# ── 3d. Hybrid: BM25+ → Embedding Re-ranking ────────────────────────────────
def run_hybrid_search(docs_df, queries_df, top_k=100,
                      bm25_candidates=HYBRID_BM25_CANDIDATES,
                      alpha=HYBRID_ALPHA,
                      model_name=EMBEDDING_MODEL,
                      batch_size=EMBEDDING_BATCH):
    """
    Two-stage hybrid retrieval:
      1. BM25+ retrieves top-N candidates (fast, lexical recall)
      2. Embedding model re-scores those candidates (semantic precision)
      3. Final score = α·norm(BM25+) + (1-α)·cos_emb
    """
    from rank_bm25 import BM25Plus
    from sentence_transformers import SentenceTransformer

    top_k = min(top_k, len(docs_df))
    bm25_candidates = max(bm25_candidates, top_k)

    # ── Stage 1: BM25+ candidate retrieval ──
    print(f"  Stage 1: BM25+ → top {bm25_candidates} candidates per query …")
    tokenized_corpus = [tokenize(text) for text in docs_df['content']]
    bm25 = BM25Plus(tokenized_corpus)
    doc_ids = docs_df['id'].to_numpy()
    doc_contents = docs_df['content'].to_numpy()

    # Collect per-query BM25 candidates and scores
    bm25_candidate_indices = []   # list of arrays, one per query
    bm25_candidate_scores = []
    for _, row in queries_df.iterrows():
        query_tokens = tokenize(row['content'])
        scores = bm25.get_scores(query_tokens)
        top_idx = np.argsort(scores)[-bm25_candidates:][::-1]
        bm25_candidate_indices.append(top_idx)
        bm25_candidate_scores.append(scores[top_idx])

    # ── Stage 2: Embedding re-scoring ──
    # Collect all unique candidate doc indices to encode only once
    unique_indices = np.unique(np.concatenate(bm25_candidate_indices))
    print(f"  Stage 2: Encoding {len(unique_indices)} unique candidate docs …")

    model = SentenceTransformer(model_name)
    candidate_texts = doc_contents[unique_indices].tolist()
    candidate_embeddings = model.encode(
        candidate_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    # Map global doc index → position in candidate_embeddings
    idx_to_emb_pos = {int(idx): pos for pos, idx in enumerate(unique_indices)}

    print(f"  Encoding {len(queries_df)} queries …")
    query_embeddings = model.encode(
        queries_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    # ── Stage 3: Score fusion ──
    results = []
    for i in range(len(queries_df)):
        cand_idx = bm25_candidate_indices[i]
        bm25_scores = bm25_candidate_scores[i]

        # Normalise BM25 scores to [0, 1]
        bm25_min = bm25_scores.min()
        bm25_max = bm25_scores.max()
        if bm25_max > bm25_min:
            bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
        else:
            bm25_norm = np.ones_like(bm25_scores)

        # Compute embedding similarity for candidates
        emb_positions = [idx_to_emb_pos[int(j)] for j in cand_idx]
        cand_embs = candidate_embeddings[emb_positions]
        emb_scores = query_embeddings[i] @ cand_embs.T  # dot product (normalised)

        # Fuse: α·BM25_norm + (1-α)·emb_sim
        fused = alpha * bm25_norm + (1 - alpha) * emb_scores

        # Re-rank by fused score and keep top_k
        rerank_order = np.argsort(fused)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[cand_idx[rerank_order]].tolist(),
        })
    return results


# ── Dispatcher ───────────────────────────────────────────────────────────────
MODELS = {
    'tfidf':     run_tfidf_search,
    'bm25':      run_bm25_search,
    'embedding': run_embedding_search,
    'hybrid':    run_hybrid_search,
}

def run_retrieval(model_name, docs_df, queries_df, top_k=100):
    if model_name not in MODELS:
        raise ValueError(f"Unknown model '{model_name}'. Choose from {list(MODELS)}")
    print(f"Running {model_name} …")
    t0 = time.time()
    results = MODELS[model_name](docs_df, queries_df, top_k=top_k)
    print(f"  Done in {time.time() - t0:.1f}s")
    return results

In [ ]:
# ── 4. Evaluation Helpers ────────────────────────────────────────────────────

def load_ground_truth(path):
    """Load qgts_train.json → {query_id: set(doc_ids)}."""
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    gt = {}
    for qid, info in raw.items():
        gt[str(qid)] = {str(d['doc_id']) for d in info['relevant_doc_ids']}
    return gt


def mean_average_precision(results, ground_truth, k=100):
    """Compute MAP@K over results that have a matching ground-truth entry."""
    aps = []
    for item in results:
        qid = str(item['query_id'])
        if qid not in ground_truth:
            continue
        relevant = ground_truth[qid]
        hits = 0
        score = 0.0
        for rank, doc_id in enumerate(item['relevant_docs'][:k], 1):
            if str(doc_id) in relevant:
                hits += 1
                score += hits / rank
        ap = score / len(relevant) if relevant else 0.0
        aps.append(ap)
    return np.mean(aps) if aps else 0.0


def recall_at_k(results, ground_truth, k=100):
    """Compute mean Recall@K."""
    recalls = []
    for item in results:
        qid = str(item['query_id'])
        if qid not in ground_truth:
            continue
        relevant = ground_truth[qid]
        retrieved = {str(d) for d in item['relevant_docs'][:k]}
        recalls.append(len(relevant & retrieved) / len(relevant) if relevant else 0.0)
    return np.mean(recalls) if recalls else 0.0


# ── 5. Kaggle CSV Writer ────────────────────────────────────────────────────

def write_kaggle_submission(results, sample_csv_path, output_csv_path):
    """Write results in exact Kaggle submission format."""
    pred_map = {
        str(item['query_id']): [str(d) for d in item['relevant_docs']]
        for item in results
    }
    with open(sample_csv_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError('Invalid sample submission format.')
    id_col = fieldnames[0]
    pred_col = fieldnames[1]
    category_col = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            qid = str(row[id_col])
            if qid not in pred_map:
                raise ValueError(f'Missing prediction for query_id: {qid}')
            out_row = {id_col: qid, pred_col: json.dumps(pred_map[qid])}
            if category_col is not None:
                out_row[category_col] = row.get(category_col, '?') or '?'
            writer.writerow(out_row)

In [ ]:
# ── 6. Load & Preprocess ─────────────────────────────────────────────────────
docs_df         = pd.read_json(DATA_DIR / 'docs.json')
test_queries_df = pd.read_json(DATA_DIR / 'queries_test.json')
train_queries_df = pd.read_json(DATA_DIR / 'queries_train.json')
sample_submission_path = DATA_DIR / 'submission.csv'
ground_truth = load_ground_truth(DATA_DIR / 'qgts_train.json')

docs_df          = create_content_column(docs_df, ['title', 'text', 'tags'])
test_queries_df  = create_content_column(test_queries_df, ['title', 'text'])
train_queries_df = create_content_column(train_queries_df, ['title', 'text'])

print(f"Documents : {len(docs_df):,}")
print(f"Train queries : {len(train_queries_df):,}")
print(f"Test queries  : {len(test_queries_df):,}")
print(f"Ground truth  : {len(ground_truth):,} queries")

Saved: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/notebooks/solutions_SeaFour.csv


## Evaluation on Training Queries

We evaluate all four retrieval methods on the 327 training queries using **MAP@100** and **Recall@100**, then choose the best model for the final test submission.

In [ ]:
# ── 7. Evaluate All Models on Train Queries ──────────────────────────────────
eval_results = {}

for model_name in ['tfidf', 'bm25', 'embedding', 'hybrid']:
    results = run_retrieval(model_name, docs_df, train_queries_df, top_k=TOP_K)
    m = mean_average_precision(results, ground_truth, k=TOP_K)
    r = recall_at_k(results, ground_truth, k=TOP_K)
    eval_results[model_name] = {'MAP@100': m, 'Recall@100': r}
    print(f"  {model_name:12s}  MAP@100 = {m:.4f}   Recall@100 = {r:.4f}")

eval_df = pd.DataFrame(eval_results).T
eval_df.index.name = 'Model'
print("\n─── Summary ───")
eval_df

## Generate Test Submission

Using the model selected by `FINAL_MODEL`, we retrieve documents for the 141 test queries and write the Kaggle submission CSV.

In [ ]:
# ── 8. Generate Final Submission ──────────────────────────────────────────────
test_results = run_retrieval(FINAL_MODEL, docs_df, test_queries_df, top_k=TOP_K)
write_kaggle_submission(test_results, sample_submission_path, OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
# ── 9. Preview Submission ─────────────────────────────────────────────────────
submission_preview = pd.read_csv(OUTPUT_PATH)
print(f"Rows: {len(submission_preview)}, Columns: {list(submission_preview.columns)}")
submission_preview.head()

,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""6b1a2049-6fc6-429d-a11f-061b10ba3507_96947"",...",?
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""da2e5c00-99e4-4e49-9be8-fd34dbe2aba9_120601""...",?
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",?
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""65a10367-e197-471c-8edc-0a73618172e1_14831"",...",?
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""927af133-bf03-4268-a3bd-94bda5c9da82_118230""...",?
